In [0]:
ECOMM_BRONZE_FILES = "abfss://bronze@gopaldatalake.dfs.core.windows.net/ecomm-gopal/"

In [0]:
# only for demo, never do in production, default size is 1 gb for optimized files
spark.conf.set(
  "spark.databricks.delta.optimize.maxFileSize",
  1048576
)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Step 1: Define explicit schema
ecomm_schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True), # the csv does not have sql datetime format
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Country", StringType(), True)
])

# LAZY , not yet started the stream
# Step 1: Read from Bronze Zone
bronze_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(ecomm_schema)
    .load(ECOMM_BRONZE_FILES)  
)

# .show will not work on stream
bronze_df.printSchema()
bronze_df.show(2)

In [0]:
%sql 
-- _ds data skipping demo
DROP TABLE IF EXISTS orderdb.gopal_ecomm_ds

In [0]:
(
bronze_df
.repartition(8)
.write
.format("delta")
.mode("overwrite")
.saveAsTable("orderdb.gopal_ecomm_ds")
)

In [0]:
%sql

DESCRIBE DETAIL orderdb.gopal_ecomm_ds

-- check numFiles, sizeInBytes

In [0]:
%sql
ALTER TABLE orderdb.gopal_ecomm_ds
SET TBLPROPERTIES(
'delta.dataSkippingStatsColumns'='StockCode,UnitPrice'
)

In [0]:
%sql

SELECT * FROM orderdb.gopal_ecomm_ds
WHERE StockCode='22749'

In [0]:
%sql
ANALYZE TABLE  orderdb.gopal_ecomm_ds COMPUTE DELTA STATISTICS

In [0]:
%sql
DESCRIBE DETAIL orderdb.gopal_ecomm_ds

In [0]:
%sql
OPTIMIZE orderdb.gopal_ecomm_ds
ZORDER BY (StockCode)

In [0]:
%sql
DESCRIBE DETAIL orderdb.gopal_ecomm_ds

In [0]:
%sql
SELECT StockCode, count(*) cnt
FROM orderdb.gopal_ecomm_ds
GROUP BY StockCode
ORDER BY cnt ASC
LIMIT 20

In [0]:
%sql
SELECT  *
FROM orderdb.gopal_ecomm_ds ds
WHERE StockCode='85049c'

In [0]:
%sql
-- for skipping data
OPTIMIZE orderdb.gopal_ecomm_sorted
ZORDER BY (StockCode)

In [0]:
%sql

SELECT DISTINCT _metadata.file_name
FROM orderdb.gopal_ecomm_sorted
WHERE StockCode='85049c'

In [0]:
%sql
SELECT * from orderdb.gopal_ecomm_ds where StockCode='22749'

In [0]:
%sql
SELECT
    COUNT(*)                         AS row_count,
    COUNT(DISTINCT InvoiceNo)        AS invoice_count,
    SUM(Quantity)                    AS total_quantity,
    AVG(UnitPrice)                   AS avg_unit_price,
    SUM(Quantity * UnitPrice)        AS sales_value
FROM orderdb.gopal_ecomm_ds
WHERE StockCode='85049C'

In [0]:

%sql
SELECT
    COUNT(*)                         AS row_count,
    COUNT(DISTINCT InvoiceNo)        AS invoice_count,
    SUM(Quantity)                    AS total_quantity,
    AVG(UnitPrice)                   AS avg_unit_price,
    SUM(Quantity * UnitPrice)        AS sales_value
FROM orderdb.gopal_ecomm_ds
WHERE StockCode='84966A'

In [0]:
%sql
SELECT StockCode, count(*) cnt
FROM orderdb.gopal_ecomm_sorted
GROUP BY StockCode
ORDER BY cnt ASC
LIMIT 20

In [0]:
spark.conf.get("spark.sql.codegen.wholeStage")

In [0]:
spark.conf.set("spark.sql.codegen.wholeStage", "false")

In [0]:
import pyspark.sql.functions as F
df = spark.range(10000000) \
    .withColumn("v", (F.col("id") % 100)) \
    .groupBy("v") \
    .count()

df.explain("formatted")

In [0]:
spark.conf.set("spark.sql.codegen.wholeStage", "true")

In [0]:
import pyspark.sql.functions as F
df = spark.range(10000000) \
    .withColumn("v", (F.col("id") % 100)) \
    .groupBy("v") \
    .count()

#df.explain("formatted")
#df.explain()
df.explain(True)
